In [ ]:
import os

SEED = 0 


os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8") 


# -------------------------------------------------------------
# now import libraries
import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

# Torch
import torch
torch.manual_seed(SEED)


import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import logging
import triku as tk 
from matplotlib import pylab
import os
import sys
import yaml
from scipy.sparse import csr_matrix
import gc
import torch
import scanit
from scipy.sparse import issparse
import scanit
from scipy.sparse import issparse
import gc
import torch

In [ ]:
nThreads = 10
import os

os.environ["OMP_NUM_THREADS"] = f"{nThreads}"
os.environ["OPENBLAS_NUM_THREADS"] = f"{nThreads}"
os.environ["MKL_NUM_THREADS"] = f"{nThreads}"
os.environ["BLIS_NUM_THREADS"] = f"{nThreads}"
os.environ["VECLIB_MAXIMUM_THREADS"] = f"{nThreads}"
os.environ["MKL_DYNAMIC"] = "FALSE"


In [ ]:

homeDir = os.getenv("HOME")

sys.path.insert(1, homeDir+"/utils/")


from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *
from _DEAplots import *
from _Aggregation import *
from _DEGs_utils import *
from _plotting import *
import rapids_singlecell as rsc
import ipynbname
import nbconvert.exporters
from nbconvert.preprocessors import TagRemovePreprocessor
import os


try:
    nb_name = ipynbname.name()
except:
    nb_name = "".join(os.path.basename(globals()['__vsc_ipynb_file__']))

print(nb_name)



In [ ]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)
DS = "B24-3455_8um_banksy"
FigTag =  "B24-3455"
DSname = "Banksy_B24-3455_8um"
base_path = "/data/Spatial_Tx" 
HashesDir = homeDir+"/hashes"
Celltype = "Tumorcells"

import cupyx.scipy.sparse
import random
from scipy import sparse
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import cupy as cp

cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
%load_ext rpy2.ipython
pd.DataFrame.iteritems = pd.DataFrame.items

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import colormaps
from matplotlib.colors import Normalize, to_hex
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pandas.api import types as ptypes


def plot_spatial_obs(
    adata,
    obs: str,
    library_id: str = "{DS}_hires_image",
    img_key: str = "hires",
    scale_key: str = "tissue_hires_scalef",
    spatial_key: str = "spatial",
    dotscale: float = 1.0,
    vmax_quantile: float = 0.99,
    vmin=None,
    vmax=None,
    marker: str = "s",
    img_alpha: float = 0.7,
    show_colorbar: bool = True,
    groups: list = None,
    linewidths: float = 0,
    width: int = 5,
    dpi: int = 100,
    ax=None,
    palette=None,
    show_legend: bool = True,
    legend_kwargs: dict = None,
    crop_image: dict = None,
    save: str = None,
):
    if legend_kwargs is None:
        legend_kwargs = {}

    # Extract image and scaling
    img = adata.uns[spatial_key][library_id]["images"][img_key]
    img_height, img_width = img.shape[:2]
    aspect = img_width / img_height

    scale = adata.uns[spatial_key][library_id]["scalefactors"][scale_key]
    diameter_px = adata.uns[spatial_key][library_id]["scalefactors"][
        "spot_diameter_fullres"
    ]

    # Extract spatial coordinates and obs values
    mask = adata.obs[obs].isin(groups) if groups is not None else adata.obs[obs].notna()
    coords = adata[mask].obsm[spatial_key] * scale
    obs_values = adata[mask].obs[obs]

    # Apply cropping if specified
    if crop_image is not None:
        xmin = int(crop_image["xmin"])
        xmax = int(crop_image["xmax"])
        ymin = int(crop_image["ymin"])
        ymax = int(crop_image["ymax"])

        img = img[ymin:ymax, xmin:xmax, ...]

        crop_mask = (
            (coords[:, 0] >= xmin)
            & (coords[:, 0] < xmax)
            & (coords[:, 1] >= ymin)
            & (coords[:, 1] < ymax)
        )

        coords = coords[crop_mask] - np.array([xmin, ymin])
        obs_values = obs_values[crop_mask]

        img_height, img_width = img.shape[:2]
        aspect = img_width / img_height

    # Figure / axis
    external_ax = ax is not None

    if not external_ax:
        fig, ax = plt.subplots(
            figsize=(width + 0.8, width / aspect),
            dpi=dpi,
        )
    else:
        fig = ax.figure

    ax.imshow(img, origin="upper", alpha=img_alpha)

    # Match plot_spatial_gene's spot sizing exactly.
    px_to_pt = 72 / dpi
    marker_size_pt2 = (diameter_px * px_to_pt / 2 * dotscale) ** 2

    if ptypes.is_numeric_dtype(obs_values):
        if vmin is None:
            vmin = 0
        if vmax is None:
            vmax = np.quantile(obs_values, vmax_quantile)

        norm = Normalize(vmin=vmin, vmax=vmax)
        cmap = colormaps.get_cmap(palette)

        sc = ax.scatter(
            coords[:, 0],
            coords[:, 1],
            c=obs_values,
            cmap=cmap,
            norm=norm,
            s=marker_size_pt2,
            linewidths=linewidths,
            edgecolors="black",
            marker=marker,
        )

        if show_colorbar:
            divider = make_axes_locatable(ax)
            cbar_ax = divider.append_axes("right", size="3%", pad=0.12)
            fig.colorbar(sc, cax=cbar_ax)
            cbar_ax.set_ylabel(
                str(obs),
                fontsize=12,
                rotation=270,
                labelpad=15,
            )
            cbar_ax.tick_params(labelsize=10)

    else:
        obs_cat = obs_values.astype("category").cat.remove_unused_categories()
        categories_present = list(obs_cat.cat.categories)

        if f"{obs}_colors" in adata.uns:
            all_categories = list(adata.obs[obs].astype("category").cat.categories)
            all_colors = [to_hex(c) for c in adata.uns[f"{obs}_colors"]]
            global_map = dict(zip(all_categories, all_colors))
            color_dict = {
                category: global_map.get(category, "#808080")
                for category in categories_present
            }
        else:
            cmap = colormaps.get_cmap(palette or "tab20")
            color_dict = {
                category: to_hex(cmap(i / max(len(categories_present) - 1, 1)))
                for i, category in enumerate(categories_present)
            }

        colors = obs_cat.map(color_dict).values

        ax.scatter(
            coords[:, 0],
            coords[:, 1],
            c=colors,
            s=marker_size_pt2,
            linewidths=linewidths,
            edgecolors="black",
            marker=marker,
        )

        if show_legend:
            handles = [
                Patch(
                    facecolor=color_dict[category],
                    edgecolor="black",
                    label=str(category),
                )
                for category in categories_present
            ]
            legend_options = {
                "loc": "center left",
                "bbox_to_anchor": (1.01, 0.5),
                "frameon": False,
            }
            legend_options.update(legend_kwargs)
            ax.legend(handles=handles, **legend_options)

    ax.axis("off")

    if not external_ax:
        plt.tight_layout()

    if save is not None:
        fig.savefig(save, dpi=dpi, bbox_inches="tight")

    if not external_ax:
        plt.show()


# Load data

In [ ]:
adata = sc.read_h5ad(f"/data/projects/spatialTX/3_Domain_characterization/{DS}_partitioned.h5ad")
adata = adata[adata.obs.spot_class.isin(["doublet_certain","singlet"])].copy()
adata


# We keep singlets and certain doublets only if above 1% of total spots

In [ ]:
adata = adata[adata.obs["spacexr"] != "reject"].copy()
#adata = adata[(~adata.obs["spacexr"].str.contains(",")) | (adata.obs["spacexr"].str.contains(",") & adata.obs["spacexr"].isin(adata.obs["spacexr"].value_counts()[adata.obs["spacexr"].value_counts() > adata.shape[0]*.01].index.tolist()))].copy()

print(adata.shape)



adata.obs["spacexr_Refined"] = adata.obs["spacexr"]

# adata.obs["spacexr_Refined"] = np.where(adata.obs["spacexr_Refined"] == "Tumorcells", "Tumorcells","Niche")
# adata.obs["spacexr_Refined"] = "Domain" + adata.obs["Spatial_Domain"].astype(str) +  "_"  +adata.obs["spacexr_Refined"].astype(str)

# We also restrict analysis to niches containing tumor for now

In [ ]:
tag ="Spatial_Domain"
frac_tumor = adata.obs["spacexr"].eq("Tumorcells").groupby(adata.obs[tag]).mean().sort_values(ascending=False).index.tolist()

assign_palette_topn_then_random(
    adata,
    obs_key="spacexr",
    palette_name="Set3",
    random_seed=123
)

print(frac_tumor)

In [ ]:
tag ="Spatial_Domain"

import yaml

with open(homeDir + "/resources/colorMaps.yaml", "r") as f:
    spacexr = yaml.load(f, Loader=yaml.FullLoader)["spacexr"]

# make categorical
adata.obs["spacexr"] = adata.obs["spacexr"].astype("category")

# sort categories by overall representation (descending)
cat_order = (
    adata.obs["spacexr"]
    .value_counts(dropna=False)
    .index
    .tolist()
)

adata.obs["spacexr"] = adata.obs["spacexr"].cat.reorder_categories(
    cat_order, ordered=True
)

# colors in the same (abundance-sorted) order
adata.uns["spacexr_colors"] = [spacexr[c] for c in adata.obs["spacexr"].cat.categories]




fig, ax, info = plot_stacked_fractions_by_cluster(
    adata,
    variable1="spacexr",
    variable2=tag,x_order_v2=frac_tumor,
    aggregate_by_variable3=False,figsize=(30,10),linewidth=0,  legend_hspace=1,legend_row_height=10,legend_cols_annotations=3,legend_fontsize=10,legend_title_fontsize=15,
    renormalize_medians=False,
)

# Restrict to niche with > 5% tumor cells

In [ ]:
minCells = adata.shape[1]
minCells = .05

TumorFractions = adata.obs["spacexr"].eq("Tumorcells").groupby(adata.obs[tag]).mean()

# Selected tumor enriched domains
SelectedDomains = TumorFractions[TumorFractions > minCells].index.tolist()
adataSS = adata[adata.obs["Spatial_Domain"].isin(SelectedDomains)].copy()
TumorFractionsSelected = TumorFractions[SelectedDomains].sort_values(ascending=False)

# Extract only tumor cells
#adataSS = adataSS[(adataSS.obs["spacexr"] == "Tumorcells")]



# Prepare contrast column
adataSS.obs["contrastCol"] = "Domain" + adataSS.obs["Spatial_Domain"].astype(str) +  "_Tumorcells"
TumorFractionsSelected.index = ["Domain"+i+"_Tumorcells"  for i in TumorFractionsSelected.index.tolist()]



print(adataSS.obs["contrastCol"].value_counts())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

counts = adataSS.obs["contrastCol"].value_counts(dropna=False)  # include NaN if you want
df = counts.rename_axis("CellType").reset_index(name="Count")
df = df.sort_values("Count", ascending=False)

n_total = df["Count"].sum()
df["Frac"] = df["Count"] / n_total

order = df["CellType"].tolist()

fig, ax = plt.subplots(figsize=(20, 6))
sns.barplot(data=df, x="CellType", y="Count", order=order, color="steelblue", ax=ax)

ymax = df["Count"].max()
ax.set_ylim(0, ymax * 1.12)

for patch, frac in zip(ax.patches, df["Frac"].to_numpy()):
    h = patch.get_height()
    ax.annotate(
        f"{frac:.1%}",
        (patch.get_x() + patch.get_width() / 2, h),
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=90,
        xytext=(0, 2),
        textcoords="offset points",
    )

ax.set_ylabel("Number of cells")
ax.set_title("Cell counts by spacexR deconvolution")
ax.tick_params(axis="x", rotation=90)
sns.despine(offset=10)

fig.tight_layout()
plt.show()


# We reassign doublets to non tumor label and pick only pure/doublets tumorcells

In [ ]:
minSpots = 100

adataSS.obs["spacexr_aggregated"] = np.where(adataSS.obs["spacexr"].str.contains(","), "Tumorcells", adataSS.obs["spacexr"])
adataSS.obs["spacexr_aggregated"].value_counts()
#adataSS = adataSS[adataSS.obs["spacexr_aggregated"] == "Tumorcells"].copy()

In [ ]:


plot_spatial_obs(
    adataSS, obs="AnnotatedDomain", width=7, dpi=300,dotscale=5 ,img_key="hires",marker='o',
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", img_alpha=.4,  legend_kwargs={"fontsize":10},# save=f"./figures/{DS}_selectedTumor_domains.svg",
)

In [ ]:
import yaml

with open(homeDir + "/resources/colorMaps.yaml", "r") as f:
    spacexr = yaml.load(f, Loader=yaml.FullLoader)["spacexr"]

# make categorical
adataSS.obs["spacexr_aggregated"] = adataSS.obs["spacexr_aggregated"].astype(str).astype("category")

# sort categories by overall representation (descending)
cat_order = (
    adataSS.obs["spacexr_aggregated"]
    .value_counts(dropna=False)
    .index
    .tolist()
)

adataSS.obs["spacexr_aggregated"] = adataSS.obs["spacexr_aggregated"].cat.reorder_categories(
    cat_order, ordered=True
)

# colors in the same (abundance-sorted) order
adataSS.uns["spacexr_aggregated_colors"] = [spacexr[c] for c in adataSS.obs["spacexr_aggregated"].cat.categories]



plot_spatial_obs(
    adataSS, obs="spacexr_aggregated", width=7, dpi=300,dotscale=3 ,img_key="hires",marker='o',
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", img_alpha=.4,  legend_kwargs={"fontsize":10}#, save=f"./figures/{DS}_selectedTumor_domain_annotation.svg",
)

# Lets group knn metaspots

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from scipy.spatial import cKDTree
from scipy import sparse
from scipy.sparse import csr_matrix
from pandas.api.types import (
    is_numeric_dtype,
    is_bool_dtype,
    is_categorical_dtype,
    is_object_dtype,
    is_string_dtype,
)


def _union_join_strings(values, sep=","):
    vals = pd.Series(values).dropna().astype(str)
    if len(vals) == 0:
        return np.nan
    parts = set()
    for v in vals:
        for p in v.split(sep):
            p = p.strip()
            if p:
                parts.add(p)
    return sep.join(sorted(parts)) if parts else np.nan


def _parse_label_series(series, sep=","):
    """
    Parse comma-separated labels into a Series of cleaned lists.
    """
    s = series.astype(str).fillna("")
    return s.str.split(sep).apply(lambda xs: [x.strip() for x in xs if str(x).strip()])


def prepare_tumor_context(
    adata,
    raw_label_key="spacexr",
    aggregated_label_key="spacexr_aggregated",
    tumor_label="Tumorcells",
    coord_key="FULLRESspatial_microns",
    max_distance=30.0,
    workers=5,
    add_flag_cols=True,
    near_prefix="near_",
    context_col="context_labels",
    in_spot_col="otherCelltypeInSpot",
):
    """
    Build a tumor-only AnnData with context annotations.

    Rules
    -----
    1. A row is considered tumor if its RAW label contains `tumor_label`.
    2. In-spot context for a tumor row is the set of non-tumor labels co-present
       in its raw label.
    3. Radius context for label `ct` is positive if any nearby row within
       `max_distance` has `ct` among its RAW comma-separated components.
       This includes:
         - pure singlets ct
         - non-tumor doublets containing ct
         - tumor-containing doublets containing ct

    Returns
    -------
    tumor_adata : AnnData
        Copy containing only tumor-containing rows.
    near_cols : list[str]
        Names of created near_* columns (empty if add_flag_cols=False).
    context_types : list[str]
        Ordered non-tumor cell types used for context.
    """
    full = adata.copy()

    raw_s = full.obs[raw_label_key].astype(str)
    parsed = _parse_label_series(raw_s)

    # tumor rows = raw label explicitly contains tumor_label
    has_tumor = parsed.apply(lambda xs: tumor_label in xs)

    # aggregated label: all tumor-containing rows become Tumorcells
    full.obs[aggregated_label_key] = np.where(has_tumor, tumor_label, raw_s)

    coords = np.ascontiguousarray(full.obsm[coord_key], dtype=np.float32)
    tumor_mask = has_tumor.to_numpy()

    if not np.any(tumor_mask):
        raise ValueError(f"No cells found whose {raw_label_key!r} contains {tumor_label!r}")

    # tumor-only object
    tumor_adata = full[tumor_mask].copy()
    tumor_coords = coords[tumor_mask]
    tumor_obs_names = full.obs_names[tumor_mask]

    # all non-tumor label components seen anywhere in RAW labels
    context_types = sorted({
        part
        for xs in parsed
        for part in xs
        if part != tumor_label
    })
    type_to_idx = {ct: i for i, ct in enumerate(context_types)}

    n_tumor = tumor_coords.shape[0]
    n_types = len(context_types)
    context_mat = np.zeros((n_tumor, n_types), dtype=np.uint8)

    # -------------------------
    # 1) in-spot context
    # -------------------------
    tumor_parsed = parsed[tumor_mask]
    in_spot = np.empty(n_tumor, dtype=object)

    for i, xs in enumerate(tumor_parsed):
        uniq = sorted({x for x in xs if x != tumor_label})
        in_spot[i] = ",".join(uniq) if uniq else np.nan

        for p in uniq:
            if p in type_to_idx:
                context_mat[i, type_to_idx[p]] = 1

    # -------------------------
    # 2) radius context
    # -------------------------
    # Use RAW membership, not aggregated labels:
    # ct is contributed by any row whose parsed raw label contains ct.
    parsed_array = parsed.to_numpy()

    for ct, j in type_to_idx.items():
        ref_mask = np.fromiter((ct in xs for xs in parsed_array), dtype=bool, count=len(parsed_array))
        ref_coords = coords[ref_mask]
        if ref_coords.shape[0] == 0:
            continue

        tree = cKDTree(ref_coords)
        neigh = tree.query_ball_point(tumor_coords, r=max_distance, workers=workers)
        context_mat[:, j] |= np.fromiter((len(x) > 0 for x in neigh), dtype=np.uint8, count=n_tumor)

    # -------------------------
    # context string per tumor row
    # -------------------------
    context_strings = np.empty(n_tumor, dtype=object)
    for i in range(n_tumor):
        labs = [context_types[j] for j in np.flatnonzero(context_mat[i])]
        context_strings[i] = ",".join(labs) if labs else tumor_label

    tumor_adata.obs[in_spot_col] = pd.Series(in_spot, index=tumor_obs_names)
    tumor_adata.obs[context_col] = pd.Series(context_strings, index=tumor_obs_names).replace("", tumor_label)

    near_cols = []
    if add_flag_cols:
        for j, ct in enumerate(context_types):
            col = f"{near_prefix}{ct}"
            tumor_adata.obs[col] = context_mat[:, j].astype(bool)
            near_cols.append(col)

    tumor_adata.obs[aggregated_label_key] = tumor_label

    return tumor_adata, near_cols, context_types


def compute_sums_by_category_fast(adata, obs_key="MyObs", blacklistCategory=None, layer=None):
    """
    Sparse group-sum of counts by adata.obs[obs_key].
    Returns DataFrame: groups x genes
    """
    groups = adata.obs[obs_key].astype(str)

    if blacklistCategory is not None:
        keep = groups != str(blacklistCategory)
        adata = adata[keep].copy()
        groups = groups[keep]

    mat = adata.X if layer is None else adata.layers[layer]
    if not sparse.issparse(mat):
        mat = sparse.csr_matrix(mat)
    else:
        mat = mat.tocsr()

    codes, uniq = pd.factorize(groups, sort=True)

    G = sparse.csr_matrix(
        (np.ones(len(codes), dtype=np.float32), (codes, np.arange(len(codes)))),
        shape=(len(uniq), adata.n_obs),
    )

    summed = G @ mat

    out = pd.DataFrame(
        summed.toarray(),
        index=pd.Index(uniq.astype(str)),
        columns=adata.var_names.astype(str),
    )
    return out.sort_index()


def compute_centroids_by_category_multi_fast(
    adata,
    obs_key,
    coord_keys=("FULLRESspatial", "spatial"),
    blacklistCategory=None,
):
    results = {}
    mappings = {}

    groups = adata.obs[obs_key].astype(str)

    if blacklistCategory is not None:
        keep = groups != str(blacklistCategory)
        adata = adata[keep].copy()
        groups = groups[keep]

    for key in coord_keys:
        coords = pd.DataFrame(
            np.asarray(adata.obsm[key]),
            index=adata.obs_names,
            columns=["x", "y"],
        )
        coords[obs_key] = groups.values

        cent = coords.groupby(obs_key, sort=True)[["x", "y"]].mean()
        cent.index = cent.index.astype(str)
        results[key] = cent.sort_index()

        mappings[key] = cent.loc[groups.astype(str)].to_numpy()

    return results, mappings


def create_anndata_from_group_summaries(
    sum_counts,
    centroids_dict,
    adata=None,
    group_key=None,
    obs_cols=None,
    blacklistCategory=None,
    uns_top_map=None,
    obs_agg=None,
):
    if uns_top_map is None:
        uns_top_map = []
    if obs_agg is None:
        obs_agg = {}

    def majority_vote(x):
        x = pd.Series(x).dropna()
        if len(x) == 0:
            return np.nan
        m = x.mode()
        return m.iloc[0] if len(m) > 0 else x.iloc[0]

    X_sparse = csr_matrix(sum_counts.values.astype("float32"))

    new_adata = ad.AnnData(X=X_sparse)
    new_adata.obs_names = sum_counts.index.astype(str)
    new_adata.var_names = sum_counts.columns.astype(str)
    new_adata.var = pd.DataFrame(index=sum_counts.columns.astype(str))

    for obsm_key, df in centroids_dict.items():
        if not sum_counts.index.equals(df.index):
            raise ValueError(f"Index mismatch between counts and centroid coordinates for `{obsm_key}`.")
        new_adata.obsm[obsm_key] = df.loc[sum_counts.index].values

    if adata is not None:
        if group_key is None:
            raise ValueError("If `adata` is provided, `group_key` must also be provided.")
        if group_key not in adata.obs.columns:
            raise KeyError(f"{group_key!r} not found in adata.obs")

        obs_df = adata.obs.copy()
        obs_df[group_key] = obs_df[group_key].astype(str)

        if blacklistCategory is not None:
            obs_df = obs_df[obs_df[group_key] != str(blacklistCategory)]

        if obs_cols is None:
            obs_cols = [c for c in obs_df.columns if c != group_key]
        else:
            missing = [c for c in obs_cols if c not in obs_df.columns]
            if missing:
                raise KeyError(f"These obs columns are missing from adata.obs: {missing}")

        def resolve_agg(col):
            if col in obs_agg:
                return obs_agg[col]
            s = obs_df[col]
            if is_bool_dtype(s):
                return "mean"
            elif is_numeric_dtype(s):
                return "mean"
            elif is_categorical_dtype(s) or is_object_dtype(s) or is_string_dtype(s):
                return majority_vote
            return majority_vote

        agg = {col: resolve_agg(col) for col in obs_cols}
        obs_grouped = obs_df.groupby(group_key, sort=False).agg(agg)
        obs_grouped.index = obs_grouped.index.astype(str)
        obs_grouped = obs_grouped.reindex(sum_counts.index.astype(str))

        for col in obs_cols:
            orig = obs_df[col]
            if is_categorical_dtype(orig):
                used = agg[col]
                if callable(used) or used in ["first"]:
                    obs_grouped[col] = pd.Categorical(obs_grouped[col], categories=orig.cat.categories)

        new_adata.obs = obs_grouped.copy()

    return new_adata

In [ ]:
adataTumor, near_cols, context_types = prepare_tumor_context(
    adataSS,
    raw_label_key="spacexr",
    aggregated_label_key="spacexr_aggregated",
    tumor_label="Tumorcells",
    coord_key="FULLRESspatial_microns",
    max_distance=50,
    workers=5,
    add_flag_cols=True,   # set False if you only want the single union string
    near_prefix="near_",
    context_col="context_labels",
    in_spot_col="otherCelltypeInSpot",
)

In [ ]:
adataTumor = adataTumor[adataTumor.obs["spacexr"] == "Tumorcells"].copy()

In [ ]:
knn = 10

add_spatial_metacells(
    adataTumor,
    new_obs=f"metacell_k{knn}",
    obsm_key="FULLRESspatial_microns",
    k=knn,
)

centroid_dict, centroids_map = compute_centroids_by_category_multi_fast(
    adataTumor,
    obs_key=f"metacell_k{knn}",
    coord_keys=["FULLRESspatial", "spatial"],
    blacklistCategory=None,
)

sum_counts = compute_sums_by_category_fast(
    adataTumor,
    obs_key=f"metacell_k{knn}",
    blacklistCategory=None,
    layer="counts",
)

assert all(df.index.equals(sum_counts.index) for df in centroid_dict.values())

In [ ]:
near_cols = [c for c in adataTumor.obs.columns if c.startswith("near_")]

adataKnn = create_anndata_from_group_summaries(
    sum_counts=sum_counts,
    centroids_dict=centroid_dict,
    adata=adataTumor,
    group_key=f"metacell_k{knn}",
    obs_cols=[
        "spacexr_Refined",
        "Spatial_Domain",
        "AnnotatedDomain",
        "spacexr",
        "otherCelltypeInSpot",
        "context_labels",
        *near_cols,
    ],
    blacklistCategory=None,
    obs_agg={
        "otherCelltypeInSpot": _union_join_strings,
        "context_labels": _union_join_strings,
        **{c: "max" for c in near_cols},
    },
)

adataKnn.layers["counts"] = adataKnn.X.copy()
adataKnn.obs_names = adataKnn.obs_names.astype(str)

adataKnn.uns["spatial"] = adataTumor.uns["spatial"]
if "FULLRESspatial" in adataTumor.uns:
    adataKnn.uns["FULLRESspatial"] = adataTumor.uns["FULLRESspatial"]

# KNN based

In [ ]:
import gc
import random
import numpy as np
import torch
import scanpy as sc
import squidpy as sq

# -----------------------------
# Reproducibility / settings
# -----------------------------
SEED = int(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

try:
    torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

tag = "Domain"

SPneigh = 20
TXneighb = 30
npcs = 15
ntop_genes = 3000

# set this to True if you want more stable PCA / variance ratio across reruns
USE_CPU_PCA = False

# -----------------------------
# Memory cleanup
# -----------------------------
print("Releasing memory")
gc.collect()
try:
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
except Exception:
    pass

# -----------------------------
# Start from counts
# -----------------------------
if "counts" not in adataKnn.layers:
    raise KeyError("adataKnn.layers['counts'] is missing")

adataKnn.X = adataKnn.layers["counts"].copy()

# clear stale analysis slots that may become inconsistent after reruns
for key in ["pca", "neighbors", "umap", "rank_genes_groups"]:
    if key in adataKnn.uns:
        del adataKnn.uns[key]

for key in ["X_pca", "X_umap"]:
    if key in adataKnn.obsm:
        del adataKnn.obsm[key]

for key in ["PCs"]:
    if key in adataKnn.varm:
        del adataKnn.varm[key]

for key in ["distances", "connectivities"]:
    if key in adataKnn.obsp:
        del adataKnn.obsp[key]

# -----------------------------
# Normalize + log on GPU
# -----------------------------
print("Moving AnnData to GPU and computing log-normalized matrix")
rsc.get.anndata_to_GPU(adataKnn)
rsc.pp.filter_genes(adataKnn, min_cells=10)
rsc.pp.normalize_total(adataKnn)
rsc.pp.log1p(adataKnn)

# freeze log-normalized expression on CPU for stable downstream use
adataKnn.layers["logCPU"] = adataKnn.X.get()

# -----------------------------
# SVGs from spatial autocorrelation
# -----------------------------
print("Computing spatial neighbors / Moran's I")
sq.gr.spatial_neighbors(adataKnn, coord_type="generic", n_neighs=SPneigh)
sq.gr.spatial_autocorr(adataKnn, mode="moran", layer="logCPU")

moran_df = adataKnn.uns["moranI"].copy()

SVGs = (
    moran_df.loc[moran_df["pval_norm_fdr_bh"] < 0.01]
    .dropna()
    .sort_values(["I", "pval_norm_fdr_bh"], ascending=[False, True])
    .head(ntop_genes)
    .index.tolist()
)

print(f"SVGs retained: {len(SVGs)}")

# -----------------------------
# PCA / neighbors
# -----------------------------
print("Scaling for PCA")

if USE_CPU_PCA:
    # use CPU for more stable PCA across reruns
    adataKnn.X = adataKnn.layers["logCPU"].copy()
    sc.pp.scale(adataKnn, zero_center=False, max_value=10)
    adataKnn.layers["scaledCPU"] = adataKnn.X.copy()

    print("Running CPU PCA")
    sc.pp.pca(adataKnn, n_comps=npcs, svd_solver="arpack", random_state=SEED, mask_var=None)
    sc.pl.pca_variance_ratio(adataKnn)

    print("Building neighbors on CPU PCA")
    sc.pp.neighbors(adataKnn, n_neighbors=TXneighb, n_pcs=npcs, random_state=SEED)

else:
    # faster, but slightly less reproducible on reruns
    adataKnn.X = adataKnn.layers["logCPU"].copy()
    rsc.get.anndata_to_GPU(adataKnn)
    rsc.pp.scale(adataKnn, zero_center=False, max_value=10)
    adataKnn.layers["scaledCPU"] = adataKnn.X.get()

    print("Running GPU PCA")
    rsc.pp.pca(adataKnn, random_state=SEED, svd_solver="covariance_eigh",mask_var=None)
    sc.pl.pca_variance_ratio(adataKnn)

    print("Building neighbors on GPU PCA")
    rsc.pp.neighbors(
        adataKnn,
        algorithm="brute",
        n_neighbors=TXneighb,
        n_pcs=npcs,
        random_state=SEED,
    )

# -----------------------------
# Triku on log-normalized data
# -----------------------------
print("Running triku on log-normalized CPU matrix")
adataKnn.X = adataKnn.layers["logCPU"].copy()

if "highly_variable" in adataKnn.var.columns:
    adataKnn.var.drop(columns=["highly_variable"], inplace=True)

tk.tl.triku(adataKnn, n_features=ntop_genes)

triku_genes = adataKnn.var_names[adataKnn.var["highly_variable"]].tolist()

# -----------------------------
# Joint variable genes
# -----------------------------
JointVGs = sorted(set(triku_genes).union(SVGs))
adataKnn.var["highly_variable"] = adataKnn.var_names.isin(JointVGs)

jvgglen = int(adataKnn.var["highly_variable"].sum())
print(f"Total joint variable genes: {jvgglen}")


# restore a sensible default X
adataKnn.X = adataKnn.layers["logCPU"].copy()

adataKnn.var["highly_variable"] = adataKnn.var_names.isin(JointVGs)
sc.pp.pca(adataKnn,random_state=SEED,  layer="scaledCPU", mask_var="highly_variable")

In [ ]:
sc.pl.pca_variance_ratio(adataKnn)

rsc.pp.neighbors(adataKnn, n_neighbors=30,algorithm="brute",n_pcs=15)
sc.tl.umap(adataKnn)

In [ ]:
sc.tl.leiden(adataKnn, flavor="igraph", resolution=.7)

In [ ]:
if "leiden_colors" in adataKnn.uns:
    del adataKnn.uns["leiden_colors"]
sc.pl.umap(adataKnn, color="leiden")

In [ ]:
plot_spatial_obs(
    adataKnn, obs="leiden", width=7, dpi=200,dotscale=4 ,img_key="hires",marker='o',linewidths=0,
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image",  img_alpha=.4,  legend_kwargs={"fontsize":10})
for leidenlabel  in adataKnn.obs["leiden"].unique():
    plot_spatial_obs(
        adataKnn, obs="leiden", width=7, dpi=200,dotscale=4 ,img_key="hires",marker='o',linewidths=0,groups=[leidenlabel],
        spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{FigTag}_hires_image", img_alpha=.4,  legend_kwargs={"fontsize":10})

# Save clustered object

In [ ]:
adataKnn.write_h5ad(f"./{DS}_KNNmetaspots.h5ad")